# ETI — first from-scratch training run (Kaggle CPU)

This notebook trains ETI's own baseline classifier from public market candles. It does **not** use a hosted trading model, broker credentials, or live orders. Kaggle Internet must be enabled so CCXT can download public candles. A GPU is not required.

The output is a research candidate artifact plus metrics. It is not automatically approved, deployed to a phone, or evidence of profitability.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib, zipfile

WORK = Path('/kaggle/working')
REPO = WORK / 'eti-research-core'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', 'https://github.com/hamza00alik-source/eti-research-core.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(REPO)
print('Repository:', REPO)
print('ETI source revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Python:', sys.version)


In [ ]:
# Install the exact repository runtime dependencies. CPU-only training is sufficient.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-mobile.txt'], check=True)
print('ETI and pinned mobile-export dependencies installed.')


## Train and evaluate

This run requests up to 3,000 completed BTC/USDT one-hour spot candles from MEXC's public API. ETI paginates requests, validates candle data, runs chronological purged walk-forward evaluation, compares against a majority-class baseline, then writes an unapproved model candidate. No exchange API key is used.

In [ ]:
os.environ['ETI_ARTIFACT_DIR'] = str(REPO / 'kaggle_output' / 'model_artifacts')
metrics_path = REPO / 'kaggle_output' / 'eti_training_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-m', 'eti.pipeline',
    '--exchange', 'mexc', '--symbol', 'BTC/USDT',
    '--timeframe', '1h', '--limit', '3000',
    '--evaluation', 'nested', '--save-model',
    '--model-id', 'eti-crypto-core', '--model-version', '0.1.0',
    '--output', str(metrics_path),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
metrics = json.loads(metrics_path.read_text())
print('\nTRAINING SUMMARY')
print(json.dumps({
    'training_performed': metrics.get('training_performed'),
    'rows': metrics.get('rows'),
    'fold_count': metrics.get('fold_count'),
    'mean_test_balanced_accuracy': metrics.get('mean_test_balanced_accuracy'),
    'candidate_summary': metrics.get('candidate_summary'),
    'cost_sensitivity': metrics.get('cost_sensitivity'),
    'trading_summary': metrics.get('trading_performance', {}).get('summary'),
    'exported_model': metrics.get('exported_model'),
    'warning': metrics.get('warning'),
}, indent=2))


In [ ]:
# Export the fitted candidate to a mobile-safe ONNX bundle and verify CPU inference parity.
artifact = metrics.get('exported_model') or {}
digest = artifact.get('artifact_sha256')
if not digest or len(digest) != 64:
    raise RuntimeError('Training did not produce a valid candidate artifact checksum.')
from eti.artifacts import ModelArtifactStore
from eti.mobile_export import export_model_to_onnx
candidate_model = ModelArtifactStore(os.environ['ETI_ARTIFACT_DIR']).load_model(artifact['artifact_uri'], digest)
mobile_result = export_model_to_onnx(
    candidate_model, artifact['input_schema']['features'], WORK / 'eti_mobile_export',
    artifact_sha256=digest,
)
mobile_bundle = Path(mobile_result['bundle_path'])
print(json.dumps({
    'onnx_sha256': mobile_result['onnx_sha256'],
    'verification': mobile_result['metadata']['verification'],
    'calibration': mobile_result['metadata']['calibration'],
    'mobile_bundle': str(mobile_bundle),
}, indent=2))


In [ ]:
# Bundle the trusted model file and its evaluation report for later mobile-export work.
artifact = metrics.get('exported_model') or {}
digest = artifact.get('artifact_sha256')
if not digest or len(digest) != 64:
    raise RuntimeError('Training did not produce a valid candidate artifact checksum.')
model_path = Path(os.environ['ETI_ARTIFACT_DIR']) / digest[:2] / f'{digest}.joblib'
if not model_path.is_file():
    raise FileNotFoundError(f'Expected candidate model artifact not found: {model_path}')
bundle = WORK / 'eti_candidate_0.1.0.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(model_path, arcname=f'model/{model_path.name}')
    archive.write(metrics_path, arcname='eti_training_metrics.json')
    archive.write(mobile_bundle, arcname='mobile/eti_mobile_model_bundle.zip')
    manifest = {
        'model_id': artifact.get('model_id'), 'version': artifact.get('version'),
        'artifact_sha256': digest, 'calibration_status': artifact.get('calibration_status'),
        'candidate_only': True, 'live_trading_enabled': False,
        'warning': 'Research candidate only. Verify checksum and out-of-sample results; never load untrusted joblib files.'
    }
    archive.writestr('manifest.json', json.dumps(manifest, indent=2))
print('Candidate bundle:', bundle)
print('Bundle size (MB):', round(bundle.stat().st_size / (1024*1024), 2))


In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(bundle)))
display(FileLink(str(metrics_path)))
display(FileLink(str(mobile_bundle)))


## What this result means

A completed run proves only that the training/evaluation pipeline executed on the downloaded sample. Review the test-fold metrics, net-of-assumed-cost simulation, data-quality report, and calibration report. If any approval gate fails, keep the model unapproved. The next engineering step is converting a validated candidate to a mobile-safe inference format; do not copy a `.joblib` file into an Android app or load artifacts from unknown sources.